# Tools and Tool Binding in LangChain
This notebook demonstrates how to define custom tools, bind them to a chat model, and orchestrate a manual tool execution loop.

### Step 1: Load Environment Credentials

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Step 2: Initialize the Base Model

In [6]:
from langchain_groq import ChatGroq

groq_model = ChatGroq(model="qwen/qwen3-32b")
response = groq_model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so the user is asking why parrots talk. I need to break this down. First, I should recall what I know about parrots and their ability to mimic human speech. Parrots are known for being vocal, right? They can imitate sounds, including human words. But why do they do this? Is it just to imitate, or is there a deeper reason?\n\nI remember reading that parrots are highly social animals. In the wild, they live in flocks and use vocalizations to communicate with each other. So maybe part of their ability to talk comes from their social nature. They learn to mimic sounds from their environment, which in the wild would help them communicate with their flock. If they\'re in captivity, they might imitate humans because that\'s their social environment now.\n\nAnother point is the structure of their vocal apparatus. Parrots have a syrinx, which is their voice box, and it\'s different from humans. They can produce a wide range of sounds, which allows them to mimic

### Step 3: Define Custom Tools & Bind Them
We use the `@tool` decorator to define tools. Type annotations and docstrings are crucial as the model uses them to determine when to call the tool.

In [7]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's very sunny in {location}"

# Bind the tool to the model
model_with_tools = groq_model.bind_tools([get_weather])

### Step 4: Invoke the Model to generate a Tool Call Request

In [8]:
response = model_with_tools.invoke("What is the weather in Chennai?")

print("Raw Response Object:")
print(response)

print("\nExtracted Tool Calls:")
for tool_call in response.tool_calls:
    print(f"Tool Name: {tool_call['name']}")
    print(f"Arguments: {tool_call['args']}")

Raw Response Object:
content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Chennai. I need to use the get_weather function. The function requires a location parameter. Chennai is a city in India, so I should specify that as the location. I\'ll call the function with "Chennai" as the argument.\n', 'tool_calls': [{'id': 'k1sv7z1g1', 'function': {'arguments': '{"location":"Chennai"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 153, 'total_tokens': 236, 'completion_time': 0.137906516, 'completion_tokens_details': {'reasoning_tokens': 58}, 'prompt_time': 0.006146964, 'prompt_tokens_details': None, 'queue_time': 0.048383946, 'total_time': 0.14405348}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f0a6b-4b69-71b3-811f-6b17232e85af-

### Step 5: Manual Tool Execution Loop
Here, we orchestrate the complete agent loop using standard message classes (`HumanMessage` and `ToolMessage`) to ensure compatibility and stability when building histories.

In [9]:
from langchain_core.messages import HumanMessage

# Step 1: User asks query (using standard HumanMessage object), model requests tool call
messages = [HumanMessage(content="What's the weather in Boston?")]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute requested tool calls and collect results
for tool_call in ai_msg.tool_calls:
    # Invoking the tool directly returns a ToolMessage object
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass messages history back to the model for the final response
final_response = model_with_tools.invoke(messages)
print("Final Model Answer:")
print(final_response.content)

Final Model Answer:
The weather in Boston is currently very sunny. Let me know if you need more details! ☀️


### Step 6: View Full Message History

In [10]:
messages

[HumanMessage(content="What's the weather in Boston?", additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call this function with "Boston" as the location. I\'ll make sure to structure the tool call correctly within the XML tags as instructed.\n', 'tool_calls': [{'id': 'z0zetq83d', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 153, 'total_tokens': 246, 'completion_time': 0.133753657, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.014457781, 'prompt_tokens_details': None, 'queue_time': 0.055555045, 'total_time': 0.148211438}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'f

### Step 7: Multiple Tools & Complex Routing (Advanced Learning)
Let's see how the model routes requests dynamically when we give it multiple tools with varying parameters.

In [11]:
@tool
def calculate_area(width: float, height: float) -> float:
    """Calculate the area of a rectangle in square meters."""
    return width * height

# Bind both tools to our model
model_with_multi_tools = groq_model.bind_tools([get_weather, calculate_area])

# Test Tool Route 1: Area calculation (requires multi-parameter inference)
res_area = model_with_multi_tools.invoke("How big is a room of width 5.5 meters and height 4.0 meters?")
print("Query 1: 'How big is a room of width 5.5m and height 4.0m?'")
print(f"Selected Tool: {res_area.tool_calls[0]['name']}")
print(f"Parsed Arguments: {res_area.tool_calls[0]['args']}\n")

# Test Tool Route 2: Weather calculation
res_weather = model_with_multi_tools.invoke("What is the weather like in New York?")
print("Query 2: 'What is the weather like in New York?'")
print(f"Selected Tool: {res_weather.tool_calls[0]['name']}")
print(f"Parsed Arguments: {res_weather.tool_calls[0]['args']}")

Query 1: 'How big is a room of width 5.5m and height 4.0m?'
Selected Tool: calculate_area
Parsed Arguments: {'height': 4, 'width': 5.5}

Query 2: 'What is the weather like in New York?'
Selected Tool: get_weather
Parsed Arguments: {'location': 'New York'}


### Step 8: Knowledge Retrieval Tool (Example)
Below we define a tool that mimics a Wikipedia query to search a small offline database. This illustrates how models can pull facts dynamically.

In [12]:
@tool
def search_wiki(topic: str) -> str:
    """Search Wikipedia for information about a given topic."""
    # Local database mock
    wiki_db = {
        "langchain": "LangChain is a popular open-source framework designed to simplify the creation of applications using LLMs.",
        "agentic ai": "Agentic AI refers to artificial intelligence agents that are capable of autonomously reasoning, planning, and invoking tools.",
        "python": "Python is a high-level general-purpose programming language known for readability and clean syntax."
    }
    topic_clean = topic.lower().strip()
    return wiki_db.get(topic_clean, f"Sorry, no entries found for '{topic}' on Wikipedia.")

# Bind all three tools
model_with_all_tools = groq_model.bind_tools([get_weather, calculate_area, search_wiki])

# Test querying wikipedia
wiki_response = model_with_all_tools.invoke("What is langchain?")
print("Wikipedia Query Response Tool Call details:")
print(wiki_response.tool_calls)

# Execute the retrieved tool call manually
if wiki_response.tool_calls:
    tool_output = search_wiki.invoke(wiki_response.tool_calls[0])
    print("\nTool Output Result:")
    print(tool_output.content)

Wikipedia Query Response Tool Call details:
[{'name': 'search_wiki', 'args': {'topic': 'langchain'}, 'id': 'f5jr53y70', 'type': 'tool_call'}]

Tool Output Result:
LangChain is a popular open-source framework designed to simplify the creation of applications using LLMs.
